In [2]:
# -----######-----###### CONCAT PKL FILES + UNIQUE IDs -----######-----###### #
import os
import pandas as pd
from tqdm import tqdm

def _concat_2908_pkls_GET_df_unique_ID(root_folder):
    """
    Traverse root_folder + subfolders, load all PKL files,
    concat into one DataFrame, and return unique rows by 'ID'.
    
    Input:
        root_folder : str  -> path to '_0_tracks_data'
    Output:
        df_unique : DataFrame with unique 'ID' rows
    """

    # Collect all .pkl file paths
    pkl_files = []
    for dirpath, _, filenames in os.walk(root_folder):
        for f in filenames:
            if f.lower().endswith(".pkl"):
                pkl_files.append(os.path.join(dirpath, f))

    if not pkl_files:
        print("⚠️ No PKL files found.")
        return pd.DataFrame()

    # Load and concat
    dfs = []
    for file in tqdm(pkl_files, desc="Loading PKL files"):
        try:
            df_tmp = pd.read_pickle(file)
            dfs.append(df_tmp)
        except Excepton as e:
            print(f"⚠️ Error reading {file}: {e}")

    if not dfs:
        print("⚠️ No valid DataFrames loaded.")
        return pd.DataFrame()

    df_all = pd.concat(dfs, ignore_index=True)

    # Drop duplicates based on ID
    if "ID" not in df_all.columns:
        raise ValueError("❌ Column 'ID' not found in DataFrames.")

    df_unique = df_all.drop_duplicates(subset=["ID"])

    # Info
    print(f"✅ Total rows: {len(df_all)}")
    print(f"🎯 Unique 'ID' rows: {len(df_unique)}")
    print(f"⚠️ Duplicate rows removed: {len(df_all) - len(df_unique)}")

    return df_unique


In [9]:
#!#!#!#!#! RUNNING STATEMENTS #!#!#!#!#!
root_folder = "_0_tracks_data"
df_unique = _concat_2908_pkls_GET_df_unique_ID(root_folder)


Loading PKL files: 100%|█████████████████████████████████████████████████████| 25/25 [00:00<00:00, 348.13it/s]

✅ Total rows: 1916
🎯 Unique 'ID' rows: 1916
⚠️ Duplicate rows removed: 0


In [10]:
# -----######-----###### AIFF COUNTER (Recursive iCloud-safe) -----######-----###### #
import os
from tqdm import tqdm

def _aiff_2908_icld_count_GET_total(root_folder, audio_extensions=None, include_placeholders=False):
    """
    Input:
      - root_folder: path to scan (str)
      - audio_extensions: list-like of extensions to count (e.g., ['.aiff', '.aif']); case-insensitive. If None → ['.aiff','.aif']
      - include_placeholders: if True, also count iCloud placeholders (*.aiff.icloud / *.aif.icloud)
    Output:
      - result dict:
          {
            'count_real': int,          # *.aiff / *.aif (not *.icloud)
            'count_placeholders': int,  # *.aiff.icloud / *.aif.icloud
            'count_total': int,         # real + (placeholders if include_placeholders)
            'scanned_files': int,       # all filenames encountered
            'scanned_dirs': int         # directories walked
          }
    Notes:
      - Skips macOS junk '._*' and '.DS_Store'
      - Matching is case-insensitive
    """

    # ---------- setup ----------
    if audio_extensions is None:
        audio_extensions = ['.aiff', '.aif']

    # Normalize and prep sets
    exts = tuple(e.lower() for e in audio_extensions)

    count_real = 0
    count_placeholders = 0
    scanned_files = 0
    scanned_dirs = 0

    # ---------- walk with TQM ----------
    # First collect all dirs to build a smooth TQM
    all_dirs = []
    for dpath, _, _ in os.walk(root_folder):
        all_dirs.append(dpath)

    for dpath in tqdm(all_dirs, desc="Scanning folders (AIFF counter)"):
        scanned_dirs += 1
        try:
            entries = os.listdir(dpath)
        except Exception:
            continue

        for name in entries:
            # Skip macOS junk
            if name.startswith('._') or name in ('.DS_Store',):
                continue

            scanned_files += 1
            low = name.lower()

            # Identify placeholders: *.ext.icloud
            is_placeholder = low.endswith('.icloud') and any(low.endswith(ext + '.icloud') for ext in exts)

            # Identify real: ends with ext and NOT .icloud
            is_real = (any(low.endswith(ext) for ext in exts) and not low.endswith('.icloud'))

            if is_real:
                count_real += 1
            elif is_placeholder:
                count_placeholders += 1

    count_total = count_real + (count_placeholders if include_placeholders else 0)

    # Summary print (so you get instant readout)
    print("===== AIFF COUNT SUMMARY =====")
    print(f"Root: {root_folder}")
    print(f"Real AIFFs (.aiff/.aif):     {count_real}")
    print(f"iCloud placeholders:         {count_placeholders}")
    print(f"TOTAL (reported):            {count_total}   {'(includes placeholders)' if include_placeholders else '(excludes placeholders)'}")
    print(f"Dirs scanned:                {scanned_dirs}")
    print(f"Files seen:                  {scanned_files}")
    print("================================")

    return {
        'count_real': count_real,
        'count_placeholders': count_placeholders,
        'count_total': count_total,
        'scanned_files': scanned_files,
        'scanned_dirs': scanned_dirs
    }


# Icloud

In [12]:
root_folder = "/Users/yerik/Library/Mobile Documents/com~apple~CloudDocs/_YDB_/_2_ms_icloud/_1_NEW_SOURCE"

# Count only real AIFF files (exclude .icloud placeholders)
res = _aiff_2908_icld_count_GET_total(
    root_folder=root_folder,
    audio_extensions=['.aiff', '.aif'],
    include_placeholders=False
)

# If you also want to include iCloud placeholders in the reported total, flip this to True:
# res = _aiff_2908_icld_count_GET_total(root_folder, ['.aiff','.aif'], include_placeholders=True)


Scanning folders (AIFF counter): 100%|██████████████████████████████████████| 78/78 [00:00<00:00, 9755.07it/s]

===== AIFF COUNT SUMMARY =====
Root: /Users/yerik/Library/Mobile Documents/com~apple~CloudDocs/_YDB_/_2_ms_icloud/_1_NEW_SOURCE
Real AIFFs (.aiff/.aif):     1916
iCloud placeholders:         0
TOTAL (reported):            1916   (excludes placeholders)
Dirs scanned:                78
Files seen:                  1993


# LOCAL

In [13]:
root_folder = "/Users/yerik/Music/_1_NEW_SOURCE"

# Count only real AIFF files (exclude .icloud placeholders)
res = _aiff_2908_icld_count_GET_total(
    root_folder=root_folder,
    audio_extensions=['.aiff', '.aif'],
    include_placeholders=False
)

# If you also want to include iCloud placeholders in the reported total, flip this to True:
# res = _aiff_2908_icld_count_GET_total(root_folder, ['.aiff','.aif'], include_placeholders=True)


Scanning folders (AIFF counter): 100%|█████████████████████████████████████| 78/78 [00:00<00:00, 10360.90it/s]

===== AIFF COUNT SUMMARY =====
Root: /Users/yerik/Music/_1_NEW_SOURCE
Real AIFFs (.aiff/.aif):     1916
iCloud placeholders:         0
TOTAL (reported):            1916   (excludes placeholders)
Dirs scanned:                78
Files seen:                  1993


# compare both folders 

In [14]:
# -----######-----###### CORE IMPORTABLE FUNCTION (Compare AIFF/WAV/etc. Files Between Two Folders) -----######-----###### #
import os
from pathlib import Path
import pandas as pd
from tqdm import tqdm

def _compare_0109_folders_GET_df_diff(
    folder_a,
    folder_b,
    exts=None
):
    """
    Compare file names in two folders (recursively).
    Returns a DataFrame with files only in A or only in B.
    
    Params:
      folder_a (str): First folder path.
      folder_b (str): Second folder path.
      exts (list or None): Optional list of extensions to filter (e.g., [".aiff", ".aif", ".wav"])
    """
    folder_a, folder_b = Path(folder_a), Path(folder_b)
    exts = [e.lower() for e in exts] if exts else None

    def collect_files(folder):
        files = []
        for f in tqdm(folder.rglob("*"), desc=f"Scanning {folder.name}"):
            if f.is_file():
                if exts is None or f.suffix.lower() in exts:
                    files.append(f.name)
        return set(files)

    set_a = collect_files(folder_a)
    set_b = collect_files(folder_b)

    only_a = sorted(set_a - set_b)
    only_b = sorted(set_b - set_a)

    df_diff = pd.DataFrame({
        "only_in_A": only_a + [""] * (max(len(only_a), len(only_b)) - len(only_a)),
        "only_in_B": only_b + [""] * (max(len(only_a), len(only_b)) - len(only_b)),
    })

    return df_diff


In [15]:
root_a = "/Users/yerik/Music/_1_NEW_SOURCE"
root_b = "/Users/yerik/Library/Mobile Documents/com~apple~CloudDocs/_YDB_/_2_ms_icloud/_1_NEW_SOURCE"

df_diff = _compare_0109_folders_GET_df_diff(root_a, root_b, exts=[".aiff", ".aif", ".wav", ".mp3"])
print(df_diff.head(30))  # show first 30 rows


Scanning _1_NEW_SOURCE: 2020it [00:00, 33782.68it/s]
Scanning _1_NEW_SOURCE: 2020it [00:00, 54286.15it/s]

Empty DataFrame
Columns: [only_in_A, only_in_B]
Index: []


# next set the whole thing about the right paths checinkg if filee exit trhough ID, and quality code

# check an ID

In [16]:
# -----######-----###### CHECK & DISPLAY ROW FOR ID -----######-----###### #
target_id = "t6-5802"

df_match = df_unique[df_unique["ID"].astype(str) == target_id]

if df_match.empty:
    print(f"❌ '{target_id}' not found in df_unique['ID']")
else:
    print(f"✅ '{target_id}' FOUND — {len(df_match)} row(s)")
    print(df_match)


✅ 't6-5802' FOUND — 1 row(s)
                                                  Path  \
564  /Users/yerik/Music/_1_NEW_SOURCE/_2023_this/_2...   

                                               temp_id  \
564  TRkw_MAMI_ARkw_Chris_Lorenzo,_COBRAH_MXkw_Exte...   

                                             file_name Extension dur_seconds  \
564  TRkw_MAMI_ARkw_Chris_Lorenzo,_COBRAH_MXkw_Exte...     .aiff  289.787347   

      dur_min     sr bit_depth bit_rate channels  ...  centroid_desc  \
564  4.829789  44100        16     None        2  ...  Muted emotion   

                 Path_png_bar_centroid                  Path_csv_freq  \
564  images/centroid_donut_t6-5802.png  tables/table_freq_t6-5802.csv   

                     Path_png_dr            Path_png_id_and_key  \
564  images/dbs_plot_t6-5802.png  images/key_and_id_t6-5802.png   

    year_written_id3 bought_year  lufs_pct  \
564      Unsupported        2023        96   

                                               re_name  

# sinc ones 

In [17]:
# -----######-----###### CONCAT MULTIPLE PKL FILES INTO ONE DF -----######-----###### #
import pandas as pd
from pathlib import Path

def _concat_2608_pkls_GET_df(folder, pkl_files):
    """
    Input:
      - folder: string, path to the directory containing .pkl files
      - pkl_files: list of .pkl filenames to concat
    Output:
      - Concatenated DataFrame
    """
    dfs = []
    for f in pkl_files:
        path = Path(folder) / f
        try:
            df = pd.read_pickle(path)
            dfs.append(df)
        except Exception as e:
            print(f"⚠️ Could not load {f}: {e}")
    
    if not dfs:
        raise ValueError("No DataFrames loaded.")
    
    df_concat = pd.concat(dfs, ignore_index=True)
    return df_concat


#!#!#!#!#! RUNNING STATEMENTS #!#!#!#!#!
folder = ""  # 🔴 change to your folder path
pkl_files = ["df_sin1t_final.pkl", "df_sinc2t_final.pkl", "df_sinc3t_final.pkl"]

df_final = _concat_2608_pkls_GET_df(folder, pkl_files)

print(df_final.shape)
#print(df_final.head())


(1261, 75)


In [18]:
import pandas as pd

# show all columns
pd.set_option("display.max_columns", None)

# show all rows (if you also want every row)
# pd.set_option("display.max_rows", None)

#print(df_final.head())   # or just df_final


In [19]:
# Check uniqueness of the 'ID' column
is_unique = df_final['ID'].is_unique
print("Are all IDs unique? ➜", is_unique)

# If not unique, see how many duplicates
if not is_unique:
    dup_count = df_final['ID'].duplicated().sum()
    print(f"⚠️ Found {dup_count} duplicate IDs")

    # Show the actual duplicate IDs
    print(df_final['ID'][df_final['ID'].duplicated()].unique())


Are all IDs unique? ➜ True


### now in folder 

In [20]:
root_folder = "/Users/yerik/Music/_3_YODJ_ADDS-inbox"

# Count only real AIFF files (exclude .icloud placeholders)
res = _aiff_2908_icld_count_GET_total(
    root_folder=root_folder,
    audio_extensions=['.mp3', '.mp3'],
    include_placeholders=False
)

# If you also want to include iCloud placeholders in the reported total, flip this to True:
# res = _aiff_2908_icld_count_GET_total(root_folder, ['.aiff','.aif'], include_placeholders=True)


Scanning folders (AIFF counter): 100%|██████████████████████████████████████| 27/27 [00:00<00:00, 8932.50it/s]

===== AIFF COUNT SUMMARY =====
Root: /Users/yerik/Music/_3_YODJ_ADDS-inbox
Real AIFFs (.aiff/.aif):     1261
iCloud placeholders:         0
TOTAL (reported):            1261   (excludes placeholders)
Dirs scanned:                27
Files seen:                  1288
